In [97]:
dataset1_drive_url = (
    "https://drive.google.com/file/d/1VTY0ff69sMqCJ_rrEgxVIteWV3Hgg6MI/view?usp=sharing"
)
dataset2_drive_url = (
    "https://drive.google.com/file/d/1qxpRjTCLh5rTJv1jHq3aXIeMxeRW5Eft/view?usp=sharing"
)

In [98]:
import re


def drive_to_direct_url(link: str) -> str:
    """
    Convert Google Drive share link to direct download URL.
    Works for multiple link formats.
    """
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'/open\?id=([a-zA-Z0-9_-]+)'
    ]
    
    file_id = None
    
    for pattern in patterns:
        match = re.search(pattern, link)
        if match:
            file_id = match.group(1)
            break
    
    if not file_id:
        raise ValueError("Invalid Google Drive link")
    
    return f"https://drive.google.com/uc?id={file_id}"

In [99]:
import pandas as pd


df1 = pd.read_csv(drive_to_direct_url(dataset1_drive_url))
df2 = pd.read_csv(drive_to_direct_url(dataset2_drive_url))

In [100]:
df1.head()

,No. of Pregnancy,Age,BMI,BP(Systolic),BP(Diastolic),DiabetesPedigreeFunction,Insulin,Skin Thickness(mm),Type-2 Diabetic,Glucose
0,3,50,22.263762,140,90.0,0,0,317.50,1,146.0
1,1,40,24.111159,110,80.0,2,0,317.50,0,61.0
2,0,21,17.183204,120,80.0,0,0,259.08,1,97.0
3,2,30,21.244332,130,85.0,0,0,322.58,1,178.0
4,2,35,22.819490,110,75.0,0,0,335.28,1,167.0


In [101]:
df2.head()

,age,gender,pulse_rate,systolic_bp,diastolic_bp,glucose,height,weight,bmi,family_diabetes,hypertensive,family_hypertension,cardiovascular_disease,stroke,diabetic
0,42,Female,66,110,73,5.88,1.65,70.2,25.75,0,0,0,0,0,No
1,35,Female,60,125,68,5.71,1.47,42.5,19.58,0,0,0,0,0,No
2,62,Female,57,127,74,6.85,1.52,47.0,20.24,0,0,0,0,0,No
3,73,Male,55,193,112,6.28,1.63,57.4,21.72,0,0,0,0,0,No
4,68,Female,71,150,81,5.71,1.42,36.0,17.79,0,0,0,0,0,No


In [102]:
df1.columns.tolist()

['No. of Pregnancy',
 'Age',
 'BMI',
 'BP(Systolic)',
 'BP(Diastolic)',
 'DiabetesPedigreeFunction',
 'Insulin',
 'Skin Thickness(mm)',
 'Type-2 Diabetic',
 'Glucose']

In [103]:
df2.columns.tolist()

['age',
 'gender',
 'pulse_rate',
 'systolic_bp',
 'diastolic_bp',
 'glucose',
 'height',
 'weight',
 'bmi',
 'family_diabetes',
 'hypertensive',
 'family_hypertension',
 'cardiovascular_disease',
 'stroke',
 'diabetic']

The two datasets will be harmonized before merging to increase the overall sample size while maintaining a consistent set of clinically meaningful features. Since the datasets contain different variables, features that are unavailable in one dataset will need to be addressed before merging. For the initial harmonization, variables that cannot be consistently represented across both datasets will be excluded rather than artificially imputed, avoiding the introduction of fabricated clinical measurements.

In `df1`, `No. of Pregnancies` will be replaced with a `gender` feature, with all records assigned `Female`, as the pregnancy information indicates that the records represent female patients. This will allow `df1` to align with the `gender` feature in `df2`.

`DiabetesPedigreeFunction` will be removed because `df2` does not contain an equivalent feature. `Insulin` will also be excluded because it is unavailable in `df2`. also `Skin Thickness(mm)`

From `df2`, `pulse_rate`, `family_diabetes`, `hypertensive`, `family_hypertension`, `cardiovascular_disease`, and `stroke` will be excluded because these features are not available in `df1`. Although some of these variables may provide useful clinical information, retaining them would result in structurally missing features for all records from `df1`. Rather than artificially imputing these values, they will be excluded to maintain consistency between the datasets.

The blood-pressure features will also be standardized by renaming `BP(Systolic)` and `BP(Diastolic)` in `df1` to `systolic_bp` and `diastolic_bp`, respectively, to match the naming convention used in `df2`. `BMI` is already available in both datasets and will remain unchanged.



In [104]:
df1["gender"] = "Female"

In [105]:
df1.drop(
    columns=[
        "No. of Pregnancy",
        "DiabetesPedigreeFunction",
        "Insulin",
        "Skin Thickness(mm)",
    ],
    inplace=True,
)

In [106]:
df1.rename(
    columns={
        "BP(Systolic)": "systolic_bp",
        "BP(Diastolic)": "diastolic_bp",
        "Type-2 Diabetic": "diabetic",
    },
    inplace=True,
)

In [107]:
df1

,Age,BMI,systolic_bp,diastolic_bp,diabetic,Glucose,gender
0,50,22.263762,140,90.0,1,146.0,Female
1,40,24.111159,110,80.0,0,61.0,Female
2,21,17.183204,120,80.0,1,97.0,Female
3,30,21.244332,130,85.0,1,178.0,Female
4,35,22.819490,110,75.0,1,167.0,Female
...,...,...,...,...,...,...,...
1060,38,22.388934,120,70.0,1,130.0,Female
1061,47,25.833385,120,80.0,1,110.0,Female
1062,50,24.972272,100,70.0,0,62.0,Female
1063,40,24.212696,110,80.0,1,125.0,Female


In [108]:
df1.columns = df1.columns.str.upper()

In [109]:
df1.columns.tolist()

['AGE', 'BMI', 'SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIABETIC', 'GLUCOSE', 'GENDER']

In [110]:
df2.drop(
    columns=[
        "pulse_rate",
        "family_diabetes",
        "hypertensive",
        "family_hypertension",
        "cardiovascular_disease",
        "height",
        "weight","stroke"
    ],
    inplace=True,
    axis=1,
)

In [111]:
df2.columns = df2.columns.str.upper()

In [112]:
df2.columns.tolist()

['AGE', 'GENDER', 'SYSTOLIC_BP', 'DIASTOLIC_BP', 'GLUCOSE', 'BMI', 'DIABETIC']

In [113]:
df1 = df1[df2.columns]

In [114]:
df1.columns.equals(df2.columns)

True

In [115]:
df = pd.concat([df1, df2], ignore_index=True)

In [116]:
df.head()

,AGE,GENDER,SYSTOLIC_BP,DIASTOLIC_BP,GLUCOSE,BMI,DIABETIC
0,50,Female,140,90.0,146.0,22.263762,1
1,40,Female,110,80.0,61.0,24.111159,0
2,21,Female,120,80.0,97.0,17.183204,1
3,30,Female,130,85.0,178.0,21.244332,1
4,35,Female,110,75.0,167.0,22.819490,1
